<a href="https://colab.research.google.com/github/ixchel-ac/secure-mobile-rag-chatbot/blob/main/fw_l1/notebooks/01_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FW-L1 Query Classifier Training

Fine-tune MobileBERT, DistilBERT, and TinyBERT to classify queries as safe or adversarial (C1-C5) for the on-device FW-L1 firewall.

**What this notebook does:**
1. Loads training data from Weave (adversarial + benign queries)
2. Trains 3 models with class-weighted loss
3. Evaluates each on held-out test set via Weave
4. Exports best model to ONNX + INT8 quantization
5. Publishes model artifact to W&B for deployment

**Labels:** `safe` (0), `C1` (1), `C2` (2), `C3` (3), `C4` (4), `C5` (5)
**Runtime:** GPU (T4) — Go to Runtime > Change runtime type > GPU
**Estimated time:** ~20-30 minutes for all 3 models



Cell 2 - Setup

In [1]:
import time
start_time = time.time()

!pip install -q transformers torch wandb weave accelerate scikit-learn onnx onnxruntime
!pip install onnx onnxruntime onnxscript

Cell 3 - Auth

In [2]:
import wandb
import weave
from google.colab import userdata

WANDB_PROJECT = "mobile-rag-firewall"

try:
    wandb_key = userdata.get("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("Logged in via Colab secrets")
except Exception:
    wandb.login()

weave.init(WANDB_PROJECT)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mariac35 (mariac35-university-of-washington) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in via Colab secrets


weave: Logged in as Weights & Biases user: mariac35.
weave: View Weave data at https://wandb.ai/mariac35-university-of-washington/mobile-rag-firewall/weave


Cell 4 - GPU check

In [3]:
import torch  # do not forget for deep learning , neural NW and tensor operations
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU: Tesla T4


Cell 5 - Load data from Weave

In [4]:
import json
from pathlib import Path
from google.colab import files

try:
    train_data = weave.ref("fw-l1-train:latest").get().rows
    val_data = weave.ref("fw-l1-val:latest").get().rows
    test_data = weave.ref("fw-l1-test:latest").get().rows
    print(f"Loaded from Weave: Train={len(train_data)}, Val={len(val_data)}, Test={len(test_data)}")
except Exception as e:
    print(f"Weave failed: {e}. Upload train.json, val.json, test.json manually.")
    uploaded = files.upload()
    with open("train.json") as f: train_data = json.load(f)
    with open("val.json") as f: val_data = json.load(f)
    with open("test.json") as f: test_data = json.load(f)

from collections import Counter
print(f"\nLabel distribution (train): {dict(Counter(ex['label'] for ex in train_data))}")

Loaded from Weave: Train=1400, Val=300, Test=300

Label distribution (train): {'C3': 140, 'C4': 140, 'safe': 700, 'C1': 140, 'C5': 140, 'C2': 140}


Cell 6 - Model setup

In [5]:
import numpy as np #help calculate prediction and metrics
from torch.utils.data import Dataset as TorchDataset #lets you create a custom dataset that PyTorch can train on
# imports transformers for hugging face used for load pretrained models,,tokenize text,
# train models, manage padding, stop training early if performance stops improving.
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.utils.class_weight import compute_class_weight

LABEL_LIST = ["safe", "C1", "C2", "C3", "C4", "C5"]
LABEL_TO_ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID_TO_LABEL = {i: l for l, i in LABEL_TO_ID.items()}
NUM_LABELS = len(LABEL_LIST)

# Models to train
# MobileBERT	Designed to be lighter/mobile-friendly
# DistilBERT Smaller and faster than BERT
# TinyBERT Very small BERT-like model

MODEL_CONFIGS = {
    "mobilebert": {"name": "google/mobilebert-uncased", "lr": 5e-5, "epochs": 10, "batch_size": 32},
    "distilbert": {"name": "distilbert-base-uncased", "lr": 5e-5, "epochs": 10, "batch_size": 32},
    "tinybert": {"name": "huawei-noah/TinyBERT_General_4L_312D", "lr": 5e-5, "epochs": 10, "batch_size": 32},
}

# creates a dataset format that Hugging Face/PyTorch can understand.
class FWL1Dataset(TorchDataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        #This converts text into numbers the model can understand.
        encoding = self.tokenizer(
            example["text"], truncation=True,
            max_length=self.max_length, padding=False,
        )
        # adds the correct label
        encoding["labels"] = example["label_id"]
        #converts everything into PyTorch tensors
        return {k: torch.tensor(v) if isinstance(v, list) else torch.tensor(v)
                for k, v in encoding.items()}

# function that calculates model performance during evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # model outputs scores for each class.
    #This line chooses the class with the highest score
    preds = np.argmax(predictions, axis=-1)
    metrics = {}
    #for each class
    for i, label_name in enumerate(LABEL_LIST):
        tp = ((preds == i) & (labels == i)).sum()
        fp = ((preds == i) & (labels != i)).sum()
        fn = ((preds != i) & (labels == i)).sum()
        # When the model predicted this class, how often was it correct?
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        # Of all real examples of this class, how many did the model find?
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        # F1 combines precision and recall into one score.
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        metrics[f"precision_{label_name}"] = float(precision)
        metrics[f"recall_{label_name}"] = float(recall)
        metrics[f"f1_{label_name}"] = float(f1)
    # Macro F1 averages the F1 score across all labels.
    metrics["f1_macro"] = float(np.mean([metrics[f"f1_{l}"] for l in LABEL_LIST]))
    #Accuracy measures total correct predictions.
    metrics["accuracy"] = float((preds == labels).mean())

    # False pass rate (adversarial classified as safe)
    # A high false pass rate means:
    # Dangerous prompts are passing through the firewall.
    adv_mask = labels != LABEL_TO_ID["safe"]
    if adv_mask.sum() > 0:
        metrics["false_pass_rate"] = float((preds[adv_mask] == LABEL_TO_ID["safe"]).sum() / adv_mask.sum())

    # False block rate (safe classified as adversarial)
    # Of all safe examples, how many were incorrectly blocked as attacks?
    # The firewall is too strict and blocks normal users.
    safe_mask = labels == LABEL_TO_ID["safe"]
    if safe_mask.sum() > 0:
        metrics["false_block_rate"] = float((preds[safe_mask] != LABEL_TO_ID["safe"]).sum() / safe_mask.sum())

    return metrics

print(f"Labels: {LABEL_LIST}")
print(f"Models: {list(MODEL_CONFIGS.keys())}")

Labels: ['safe', 'C1', 'C2', 'C3', 'C4', 'C5']
Models: ['mobilebert', 'distilbert', 'tinybert']


Cell 7 - Trainning Function

Choose model → prepare data → train → evaluate → save best model → log results

In [6]:
def train_model(model_key, train_data, val_data):
    # Load model settings
    config = MODEL_CONFIGS[model_key]
    model_name = config["name"]
    output_dir = f"models/{model_key}"

    # print training info
    print(f"\n{'=' * 60}")
    print(f"  Training: {model_key} ({model_name})")
    print(f"  Train: {len(train_data)}, Val: {len(val_data)}")
    print(f"{'=' * 60}")

    # Class weights for imbalanced data (1000 safe vs 200 per Class)
    all_labels = [ex["label_id"] for ex in train_data]
    class_weights = compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=all_labels)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
    print(f"  Class weights: {dict(zip(LABEL_LIST, class_weights.round(3)))}")

    # Start Weights & Biases logging/records
    wandb.init(
        project=WANDB_PROJECT,
        name=f"fw-l1-{model_key}",
        config={"model": model_name, "model_key": model_key,
                "learning_rate": config["lr"], "epochs": config["epochs"],
                "batch_size": config["batch_size"],
                "train_size": len(train_data), "val_size": len(val_data),
                "labels": LABEL_LIST, "class_weights": class_weights.tolist()},
        tags=["fw-l1", model_key, "colab"],
    )

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=NUM_LABELS,
        id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
    )

    #
    # The tokenizer converts text into model input numbers.
    train_dataset = FWL1Dataset(train_data, tokenizer)
    val_dataset = FWL1Dataset(val_data, tokenizer)

    # Handles padding dynamically.
    # That means shorter texts are padded only as much as needed inside each batch.
    data_collator = DataCollatorWithPadding(tokenizer)

    # Custom trainer with weighted loss
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            # loss function uses class weights so minority
            # classes matter more during training
            loss_fn = torch.nn.CrossEntropyLoss(
                weight=class_weights_tensor.to(outputs.logits.device)
            )
            loss = loss_fn(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss

    training_args = TrainingArguments(
        output_dir=output_dir, run_name=f"fw-l1-{model_key}",
        report_to="wandb",
        num_train_epochs=config["epochs"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=config["batch_size"],
        learning_rate=config["lr"], weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        greater_is_better=True, save_total_limit=1,
        logging_steps=50, fp16=torch.cuda.is_available(),
        dataloader_num_workers=2, dataloader_pin_memory=True,
    )

    trainer = WeightedTrainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=val_dataset,
        data_collator=data_collator, compute_metrics=compute_metrics,
        #If validation performance does not improve for 3 epochs, stop early.
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    # train → evaluate → save → repeat
    trainer.train()

    # Save the best model
    best_dir = f"{output_dir}/best"
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)

    # evaluates the best loaded model on validation data.
    eval_results = trainer.evaluate()
    print(f"\n  Final results for {model_key}:")
    for k, v in eval_results.items():
        if isinstance(v, float):
            print(f"    {k}: {v:.4f}")

    # saves final metrics in Weights & Biases and closes the logging run
    wandb.log({f"final/{k}": v for k, v in eval_results.items()})
    wandb.finish()
    return eval_results

print("Training function ready.")

Training function ready.


Cell 8 - Train all 3 models

Train 3 models → collect results → compare side-by-side

Good model should have:

High F1 Macro
→ balanced across all attack types

Low False Pass Rate (FPR)
→ fewer attacks slipping through

Low False Block Rate (FBR)
→ fewer safe queries blocked

In [7]:
#Empty dictionary to store results from each model
all_results = {}

# For each model: Calls your training function
# Trains the model
# Gets evaluation results
# Stores them
for model_key in ["mobilebert", "distilbert", "tinybert"]:
    results = train_model(model_key, train_data, val_data)
    all_results[model_key] = results

print(f"\n{'=' * 70}")
print(f"  {'Model':<15} {'F1 Macro':>10} {'Accuracy':>10} {'FPR':>10} {'FBR':>10}")
print(f"  {'-'*15} {'-'*10} {'-'*10} {'-'*10} {'-'*10}")
for key, r in all_results.items():
    f1 = r.get("eval_f1_macro", 0)
    acc = r.get("eval_accuracy", 0)
    fpr = r.get("eval_false_pass_rate", 0)
    fbr = r.get("eval_false_block_rate", 0)
    print(f"  {key:<15} {f1:>10.4f} {acc:>10.4f} {fpr:>10.4f} {fbr:>10.4f}")


  Training: mobilebert (google/mobilebert-uncased)
  Train: 1400, Val: 300
  Class weights: {'safe': np.float64(0.333), 'C1': np.float64(1.667), 'C2': np.float64(1.667), 'C3': np.float64(1.667), 'C4': np.float64(1.667), 'C5': np.float64(1.667)}


wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: mariac35.
weave: View Weave data at https://wandb.ai/mariac35-university-of-washington/mobile-rag-firewall/weave
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1111 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias'].
There were unexpected keys in the checkp

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Final results for tinybert:
    eval_loss: 0.4196
    eval_precision_safe: 0.9801
    eval_recall_safe: 0.9867
    eval_f1_safe: 0.9834
    eval_precision_C1: 0.9667
    eval_recall_C1: 0.9667
    eval_f1_C1: 0.9667
    eval_precision_C2: 0.9655
    eval_recall_C2: 0.9333
    eval_f1_C2: 0.9492
    eval_precision_C3: 0.9333
    eval_recall_C3: 0.9333
    eval_f1_C3: 0.9333
    eval_precision_C4: 0.9667
    eval_recall_C4: 0.9667
    eval_f1_C4: 0.9667
    eval_precision_C5: 0.9000
    eval_recall_C5: 0.9000
    eval_f1_C5: 0.9000
    eval_f1_macro: 0.9499
    eval_accuracy: 0.9633
    eval_false_pass_rate: 0.0200
    eval_false_block_rate: 0.0133
    eval_runtime: 0.3637
    eval_samples_per_second: 824.9230
    eval_steps_per_second: 27.4970
    epoch: 10.0000


eval/accuracy,▁▂▄▅▆▇▇████
eval/f1_C1,▁▄▇████████
eval/f1_C2,▁▂▅▅▆█▆▇▇▇▇
eval/f1_C3,▁▃▇████████
eval/f1_C4,▅▁▆████████
eval/f1_C5,▁▂▁▂▅▇▇████
eval/f1_macro,▁▂▄▅▆▇▇████
eval/f1_safe,▁▃▄▄▅▆▇▇███
eval/false_block_rate,▁▆████▃▆▆▆▆
eval/false_pass_rate,█▅▅▄▄▂▂▁▁▁▁
+48,...



  Model             F1 Macro   Accuracy        FPR        FBR
  --------------- ---------- ---------- ---------- ----------
  mobilebert          0.1111     0.5000     1.0000     0.0000
  distilbert          0.9833     0.9900     0.0000     0.0000
  tinybert            0.9499     0.9633     0.0200     0.0133


Cell 9 - weave evaluation on test set

Load saved best model → run predictions on test data → score results in Weave


In [8]:
from transformers import pipeline as hf_pipeline

#wraps the trained model so Weave can evaluate it.
class FWL1Model(weave.Model):
    model_key: str = ""
    model_path: str = ""
    _pipeline: object = None #the actual Hugging Face classifier

    # Only load the model when it is first needed.
    def _ensure_loaded(self):
        if self._pipeline is None:
            self._pipeline = hf_pipeline(
                "text-classification", model=self.model_path,
                tokenizer=self.model_path, top_k=None,
            )

    @weave.op
    def predict(self, text: str, label: str, label_id: int, expected_action: str) -> dict:
        self._ensure_loaded()
        # sends the text into the trained classifier
        results = self._pipeline(text)
        # changes the output into
        scores = {r["label"]: r["score"] for r in results[0]}

        pred_scores = {}
        for i, label_name in enumerate(LABEL_LIST):
            pred_scores[label_name] = scores.get(f"LABEL_{i}", scores.get(label_name, 0))
        # selects the label with the highest score
        pred_label_id = max(pred_scores, key=pred_scores.get)

        #converts classification into firewall behavior
        pred_action = "allow" if pred_label_id == "safe" else "block"

        # Return prediction information
        return {
            "predicted_label": pred_label_id,
            "predicted_action": pred_action,
            "confidence": pred_scores[pred_label_id],
            "true_label": label,
            "true_action": expected_action,
            "correct": pred_label_id == label,
            "scores": pred_scores,
        }

# tells Weave how to score each prediction
@weave.op
def classification_scorer(output: dict) -> dict:
    correct = output["correct"]
    #An attack should be blocked, but the model allowed it.
    is_false_pass = (output["true_action"] == "block" and output["predicted_action"] == "allow")
    #A safe prompt should be allowed, but the model blocked it.
    is_false_block = (output["true_action"] == "allow" and output["predicted_action"] == "block")

    return {
        "correct": 1.0 if correct else 0.0,
        "false_pass": 1.0 if is_false_pass else 0.0,
        "false_block": 1.0 if is_false_block else 0.0,
    }

# creates a Weave dataset from your test data and publishes it
test_dataset = weave.Dataset(name="fw-l1-test", rows=test_data)
weave.publish(test_dataset)

#Evaluate each saved model
for model_key in ["mobilebert", "distilbert", "tinybert"]:
    model_path = f"models/{model_key}/best"
    if not Path(model_path).exists():
        continue
    # prepares the trained model for Weave evaluation.
    model = FWL1Model(name=f"fw-l1-{model_key}", model_key=model_key, model_path=model_path)
    # Evaluate this model on the test dataset using this scorer.
    evaluation = weave.Evaluation(
        name=f"fw-l1-eval-{model_key}",
        dataset=test_dataset,
        scorers=[classification_scorer],
    )
    # runs the model on every test example
    results = await evaluation.evaluate(model)

    print(f"  {model_key}: {results}")

weave: 📦 Published to https://wandb.ai/mariac35-university-of-washington/mobile-rag-firewall/weave/objects/fw-l1-test/versions/dprYtspxWNjKYuic0zp8MY0xGcFyQLLkwH8yFeNhmsM
weave: 🍩 https://wandb.ai/mariac35-university-of-washington/mobile-rag-firewall/r/call/019e0054-dedb-7e5b-b9b9-72277a54391d


Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]

weave: 🍩 https://wandb.ai/mariac35-university-of-washington/mobile-rag-firewall/r/call/019e0055-a09e-7035-a5d5-0a1f60030dca
weave: Evaluated 1 of 300 examples
weave: Evaluated 2 of 300 examples
weave: Evaluated 3 of 300 examples
weave: Evaluated 4 of 300 examples
weave: Evaluated 5 of 300 examples
weave: Evaluated 6 of 300 examples
weave: Evaluated 7 of 300 examples
weave: Evaluated 8 of 300 examples
weave: Evaluated 9 of 300 examples
weave: Evaluated 10 of 300 examples
weave: Evaluated 11 of 300 examples
weave: Evaluated 12 of 300 examples
weave: Evaluated 13 of 300 examples
weave: Evaluated 14 of 300 examples
weave: Evaluated 15 of 300 examples
weave: Evaluated 16 of 300 examples
weave: Evaluated 17 of 300 examples
weave: Evaluated 18 of 300 examples
weave: Evaluated 19 of 300 examples
weave: Evaluated 20 of 300 examples
weave: Evaluated 21 of 300 examples
weave: Evaluated 22 of 300 examples
weave: Evaluated 23 of 300 examples
weave: Evaluated 24 of 300 examples
weave: Evaluated 25 o

  tinybert: {'output': {'confidence': {'mean': 0.686752789914608}, 'correct': {'true_count': 287, 'true_fraction': 0.9566666666666667}, 'scores': {'safe': {'mean': 0.3469867220831414}, 'C1': {'mean': 0.10924977542522053}, 'C2': {'mean': 0.13565292864727477}, 'C3': {'mean': 0.11294044999716182}, 'C4': {'mean': 0.1126513214999189}, 'C5': {'mean': 0.18251880388706923}}}, 'classification_scorer': {'correct': {'mean': 0.9566666666666667}, 'false_pass': {'mean': 0.0033333333333333335}, 'false_block': {'mean': 0.006666666666666667}}, 'model_latency': {'mean': 0.2712536819775899}}


Cell 10 - ONNX export + INT8 quantization

This cell converts your best trained FW-L1 model into a smaller deployment format.

Load best PyTorch model → export to ONNX → quantize to INT8 → validate → save tokenizer

In [10]:
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
# Remember:
# quantize_dynamic = makes the model smaller and faster
# QuantType.QInt8 = uses 8-bit integer weights

BEST_MODEL = "tinybert"  # Best model based on training results (F1 macro 0.98, accuracy 0.99)
model_path = f"models/{BEST_MODEL}/best"
onnx_dir = Path("models/onnx")
onnx_dir.mkdir(parents=True, exist_ok=True)

# 1. Load PyTorch model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

# 2. Export to ONNX — force legacy exporter (dynamo=False) for onnxruntime compatibility
dummy = tokenizer("What medications is the patient taking?",
                   return_tensors="pt", max_length=128, truncation=True)
onnx_path = onnx_dir / "fw_l1_fp32.onnx"

with torch.no_grad():
    torch.onnx.export(
        model, (dummy["input_ids"], dummy["attention_mask"]),
        str(onnx_path),
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids":      {0: "batch", 1: "seq_len"},
            "attention_mask": {0: "batch", 1: "seq_len"},
            "logits":         {0: "batch"},
        },
        opset_version=18,
        do_constant_folding=True,
        dynamo=False,  # force legacy exporter — dynamo graph is incompatible with onnxruntime quantizer
    )
print(f"Exported ONNX (FP32): {onnx_path} ({onnx_path.stat().st_size / 1024**2:.1f} MB)")

# 3. INT8 quantization
quantized_path = onnx_dir / "fw_l1.onnx"
quantize_dynamic(
    model_input=str(onnx_path),
    model_output=str(quantized_path),
    weight_type=QuantType.QInt8,
)
print(f"Quantized ONNX (INT8): {quantized_path} ({quantized_path.stat().st_size / 1024**2:.1f} MB)")

# 4. Validate ONNX matches PyTorch
import onnxruntime as ort

session = ort.InferenceSession(str(quantized_path))
onnx_out = session.run(None, {
    "input_ids": dummy["input_ids"].numpy(),
    "attention_mask": dummy["attention_mask"].numpy(),
})[0]

with torch.no_grad():
    pt_out = model(**dummy).logits.numpy()

diff = np.abs(onnx_out - pt_out).max()
print(f"Max output difference (ONNX vs PyTorch): {diff:.6f}")
assert diff < 1.0, f"ONNX/PyTorch mismatch too large: {diff}"
print("ONNX validation passed!")

# 5. Save tokenizer alongside ONNX
tokenizer.save_pretrained(str(onnx_dir / "tokenizer"))
print(f"Tokenizer saved to: {onnx_dir / 'tokenizer'}")

Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]

/tmp/ipykernel_67431/2290892595.py:23: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/masking_utils.py:171: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolea

Exported ONNX (FP32): models/onnx/fw_l1_fp32.onnx (54.8 MB)
Quantized ONNX (INT8): models/onnx/fw_l1.onnx (13.9 MB)
Max output difference (ONNX vs PyTorch): 0.477398
ONNX validation passed!
Tokenizer saved to: models/onnx/tokenizer


Cell 11 — Publish to W&B

In [11]:
run = wandb.init(
    project=WANDB_PROJECT,
    name=f"publish-fw-l1-{BEST_MODEL}",
    job_type="publish-model",
    tags=["fw-l1", BEST_MODEL, "onnx", "publish"],
)

# ONNX model + tokenizer
artifact = wandb.Artifact(
    name="fw-l1-model", type="model",
    description=f"FW-L1 query classifier ({BEST_MODEL}, INT8 ONNX) for on-device deployment",
    metadata={"model_key": BEST_MODEL, "format": "onnx_int8"},
)
artifact.add_file(str(onnx_dir / "fw_l1.onnx"))
artifact.add_dir(str(onnx_dir / "tokenizer"), name="tokenizer")
run.log_artifact(artifact)

# PyTorch model (for backend fallback)
pt_artifact = wandb.Artifact(
    name="fw-l1-model-pytorch", type="model",
    description=f"FW-L1 query classifier ({BEST_MODEL}, PyTorch) for backend use",
    metadata={"model_key": BEST_MODEL, "format": "pytorch"},
)
pt_artifact.add_dir(model_path)
run.log_artifact(pt_artifact)

run.finish()
print(f"Published: fw-l1-model (ONNX) and fw-l1-model-pytorch (PyTorch)")

wandb: Initializing weave.


Output()

weave: Logged in as Weights & Biases user: mariac35.
weave: View Weave data at https://wandb.ai/mariac35-university-of-washington/mobile-rag-firewall/weave
wandb: Adding directory to artifact (models/onnx/tokenizer)... Done. 0.0s
wandb: Adding directory to artifact (models/tinybert/best)... Done. 0.2s


Published: fw-l1-model (ONNX) and fw-l1-model-pytorch (PyTorch)


Cell 12 - Timing

In [12]:
end_time = time.time()
mins, secs = divmod(end_time - start_time, 60)
print(f"Total Notebook Execution Time: {int(mins)}m {int(secs)}s")

Total Notebook Execution Time: 10m 31s
